In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("output/basic_with_cohort_unfiltered.csv")

features = df.columns.difference(['country_code', 'year', 'Population_total'])

pop_pivot = df.pivot(index='country_code', columns='year', values='Population_total')
features_pivot = df.pivot(index='country_code', columns='year', values=features.tolist())

pop_pivot.columns = [f'Population_{col}' for col in pop_pivot.columns]
features_pivot.columns = [f'{feat}_{year}' for feat, year in features_pivot.columns]

X = pd.concat([pop_pivot, features_pivot], axis=1)

In [10]:
X=X.reset_index()
print(X['country_code'].head(2))
print(X.columns)
print(X.head(2))
X.to_csv("output/wide_unfiltered.csv", index = False)

0    AFE
1    AFG
Name: country_code, dtype: object
Index(['country_code', 'Population_1960', 'Population_1961', 'Population_1962',
       'Population_1963', 'Population_1964', 'Population_1965',
       'Population_1966', 'Population_1967', 'Population_1968',
       ...
       'mort_95_99_2014', 'mort_95_99_2015', 'mort_95_99_2016',
       'mort_95_99_2017', 'mort_95_99_2018', 'mort_95_99_2019',
       'mort_95_99_2020', 'mort_95_99_2021', 'mort_95_99_2022',
       'mort_95_99_2023'],
      dtype='object', length=6401)
  country_code  Population_1960  Population_1961  Population_1962  \
0          AFE      130075728.0      133534923.0      137171659.0   
1          AFG        9035043.0        9214083.0        9404406.0   

   Population_1963  Population_1964  Population_1965  Population_1966  \
0      140945536.0      144904094.0      149033472.0      153281203.0   
1        9604487.0        9814318.0       10036008.0       10266395.0   

   Population_1967  Population_1968  ...  mort_

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("output/basic_with_cohort_filtered.csv")

features = df.columns.difference(['country_code', 'year', 'Population_total'])

pop_pivot = df.pivot(index='country_code', columns='year', values='Population_total')
features_pivot = df.pivot(index='country_code', columns='year', values=features.tolist())

pop_pivot.columns = [f'Population_{col}' for col in pop_pivot.columns]
features_pivot.columns = [f'{feat}_{year}' for feat, year in features_pivot.columns]

X = pd.concat([pop_pivot, features_pivot], axis=1)
X=X.reset_index()
X.to_csv("output/wide_filtered.csv", index = False)

In [ ]:
import pandas as pd

df = pd.read_csv("output/basic_with_cohort_filtered.csv")

just_pop = df[["country_code", "year", "Population_total"]].copy()
just_pop.to_csv("final/just_pop.csv", index=False)
print(just_pop.tail(5))

      country_code  year  Population_total
13243          ZWE  2019        15271368.0
13244          ZWE  2020        15526888.0
13245          ZWE  2021        15797210.0
13246          ZWE  2022        16069056.0
13247          ZWE  2023        16340822.0


In [38]:
import pandas as pd

df = pd.read_csv("output/basic_with_cohort_filtered.csv")
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

DATA_RANGE = 25
PREDICT_RANGE = 10
DATA_LAST_YEAR = 2022

df = df.sort_values(["country_code", "year"])

res_list = []

for shift in list(range(0, DATA_RANGE + 1)) + [DATA_RANGE + PREDICT_RANGE]:
    df_shift = df[["country_code", "year", "Population_total"]].copy()
    df_shift["year"] = df_shift["year"] - shift
    df_shift = df_shift.rename(columns={"Population_total": f"pop_plus_{shift}"})
    res_list.append(df_shift)

res = res_list[0]
for d in res_list[1:]:
    res = res.merge(d, on=["country_code", "year"], how="left")

res = res[(res["year"] >= 1960) & 
          (res["year"] <= DATA_LAST_YEAR - DATA_RANGE - PREDICT_RANGE)]

res.to_csv("final/filtered_25_predict10_pop.csv", index=False)
print(res.head())


  country_code  year   pop_plus_0   pop_plus_1   pop_plus_2   pop_plus_3  \
0          AFE  1960  130075728.0  133534923.0  137171659.0  140945536.0   
1          AFE  1961  133534923.0  137171659.0  140945536.0  144904094.0   
2          AFE  1962  137171659.0  140945536.0  144904094.0  149033472.0   
3          AFE  1963  140945536.0  144904094.0  149033472.0  153281203.0   
4          AFE  1964  144904094.0  149033472.0  153281203.0  157704381.0   

    pop_plus_4   pop_plus_5   pop_plus_6   pop_plus_7  ...  pop_plus_17  \
0  144904094.0  149033472.0  153281203.0  157704381.0  ...  210680842.0   
1  149033472.0  153281203.0  157704381.0  162329396.0  ...  217074286.0   
2  153281203.0  157704381.0  162329396.0  167088245.0  ...  223974122.0   
3  157704381.0  162329396.0  167088245.0  171984985.0  ...  230792729.0   
4  162329396.0  167088245.0  171984985.0  177022314.0  ...  238043099.0   

   pop_plus_18  pop_plus_19  pop_plus_20  pop_plus_21  pop_plus_22  \
0  217074286.0  223974

In [35]:
import pandas as pd

df = pd.read_csv("output/basic_with_cohort_filtered.csv")
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
DATA_RANGE = 25
PREDICT_RANGE = 10
DATA_LAST_YEAR = 2022

cols = [c for c in df.columns if c not in ["country_code", "year"]]

df = df.sort_values(["country_code", "year"])

res = df[["country_code", "year"]].drop_duplicates().copy()

for col in cols:
    df_list = []
    for shift in list(range(0, DATA_RANGE + 1)) + [DATA_RANGE+PREDICT_RANGE]:
        if (shift==DATA_RANGE+PREDICT_RANGE and col != "Population_total"):
            continue
        df_shift = df[["country_code", "year", col]].copy()
        df_shift["year"] = df_shift["year"] - shift
        if col == "Population_total":
            new_name = f"pop_plus_{shift}"
        else:
            new_name = f"{col}_plus_{shift}"
        df_shift = df_shift.rename(columns={col:new_name})
        df_list.append(df_shift)
    block = df_list[0]
    for d in df_list[1:]:
        block = block.merge(d, on=["country_code", "year"], how="left")

    res = res.merge(block, on=["country_code", "year"], how="left")

res = res[(res["year"] >= 1960) &
          (res["year"] <= DATA_LAST_YEAR - DATA_RANGE - PREDICT_RANGE)]

res.to_csv("final/filtered_25_predict10_all.csv", index=False)


In [36]:
print(res.columns)

Index(['country_code', 'year', 'Pop_density_plus_0', 'Pop_density_plus_1',
       'Pop_density_plus_2', 'Pop_density_plus_3', 'Pop_density_plus_4',
       'Pop_density_plus_5', 'Pop_density_plus_6', 'Pop_density_plus_7',
       ...
       'POP100PLUS_plus_16', 'POP100PLUS_plus_17', 'POP100PLUS_plus_18',
       'POP100PLUS_plus_19', 'POP100PLUS_plus_20', 'POP100PLUS_plus_21',
       'POP100PLUS_plus_22', 'POP100PLUS_plus_23', 'POP100PLUS_plus_24',
       'POP100PLUS_plus_25'],
      dtype='object', length=1823)


In [39]:
import pandas as pd

df = pd.read_csv("output/basic_with_cohort_filtered.csv",  index_col=False)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
DATA_RANGE = 25
PREDICT_RANGE = 10
DATA_LAST_YEAR = 2022

exclude_prefixes = ["POP", "mort_", "asfr_"]

cols = [c for c in df.columns if c not in ["country_code", "year"] 
        and not any(c.startswith(p) for p in exclude_prefixes)]

df = df.sort_values(["country_code", "year"])

res = df[["country_code", "year"]].drop_duplicates().copy()

for col in cols:
    df_list = []
    for shift in list(range(0, DATA_RANGE + 1)) + [DATA_RANGE + PREDICT_RANGE]:
        if (shift==DATA_RANGE+PREDICT_RANGE and col != "Population_total"):
            continue
        df_shift = df[["country_code", "year", col]].copy()
        df_shift["year"] = df_shift["year"] - shift
        if col == "Population_total":
            new_name = f"pop_plus_{shift}"
        else:
            new_name = f"{col}_plus_{shift}"
        df_shift = df_shift.rename(columns={col: new_name})
        df_list.append(df_shift)

    block = df_list[0]
    for d in df_list[1:]:
        block = block.merge(d, on=["country_code", "year"], how="left")

    res = res.merge(block, on=["country_code", "year"], how="left")

res = res[(res["year"] >= 1960) &
          (res["year"] <= DATA_LAST_YEAR - DATA_RANGE - PREDICT_RANGE)]
print(res.columns)
res.to_csv("final/filtered_25_predict10_noncohort.csv", index=False)


Index(['country_code', 'year', 'Pop_density_plus_0', 'Pop_density_plus_1',
       'Pop_density_plus_2', 'Pop_density_plus_3', 'Pop_density_plus_4',
       'Pop_density_plus_5', 'Pop_density_plus_6', 'Pop_density_plus_7',
       ...
       'Urban_population_percent_plus_16', 'Urban_population_percent_plus_17',
       'Urban_population_percent_plus_18', 'Urban_population_percent_plus_19',
       'Urban_population_percent_plus_20', 'Urban_population_percent_plus_21',
       'Urban_population_percent_plus_22', 'Urban_population_percent_plus_23',
       'Urban_population_percent_plus_24', 'Urban_population_percent_plus_25'],
      dtype='object', length=497)


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("output/basic_with_cohort_filtered.csv", index_col=False)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Фильтруем только Россию
df = df[df["country_code"] == "RUS"].copy()

DATA_RANGE = 25
PREDICT_RANGE = 10
DATA_LAST_YEAR = 2032

cols = [c for c in df.columns if c not in ["country_code", "year"]]
df = df.sort_values(["country_code", "year"])

res = df[["country_code", "year"]].drop_duplicates().copy()

for col in cols:
    df_list = []
    for shift in list(range(0, DATA_RANGE + 1)) + [DATA_RANGE + PREDICT_RANGE]:
        if (shift == DATA_RANGE + PREDICT_RANGE and col != "Population_total"):
            continue
        df_shift = df[["country_code", "year", col]].copy()
        
        # Сохраняем исходный год для проверки
        original_year = df_shift["year"].copy()
        
        # Сдвигаем год
        df_shift["year"] = df_shift["year"] - shift
        
        # Для Population_total: если исходный год > 2022, ставим NaN
        if col == "Population_total":
            new_name = f"pop_plus_{shift}"
            # Проверяем, не превышает ли исходный год 2022
            df_shift.loc[original_year > DATA_LAST_YEAR, col] = np.nan
        else:
            new_name = f"{col}_plus_{shift}"
        
        df_shift = df_shift.rename(columns={col: new_name})
        df_list.append(df_shift)

    block = df_list[0]
    for d in df_list[1:]:
        block = block.merge(d, on=["country_code", "year"], how="left")

    res = res.merge(block, on=["country_code", "year"], how="left")

res = res[(res["year"] >= 1960) &
          (res["year"] <= DATA_LAST_YEAR - DATA_RANGE - PREDICT_RANGE)]

print(f"Количество строк: {len(res)}")
print(f"Количество колонок: {len(res.columns)}")
print(res.columns)
res.to_csv("final/russia_25_predict10.csv", index=False)


Количество строк: 38
Количество колонок: 1823
Index(['country_code', 'year', 'Pop_density_plus_0', 'Pop_density_plus_1',
       'Pop_density_plus_2', 'Pop_density_plus_3', 'Pop_density_plus_4',
       'Pop_density_plus_5', 'Pop_density_plus_6', 'Pop_density_plus_7',
       ...
       'POP100PLUS_plus_16', 'POP100PLUS_plus_17', 'POP100PLUS_plus_18',
       'POP100PLUS_plus_19', 'POP100PLUS_plus_20', 'POP100PLUS_plus_21',
       'POP100PLUS_plus_22', 'POP100PLUS_plus_23', 'POP100PLUS_plus_24',
       'POP100PLUS_plus_25'],
      dtype='object', length=1823)
